# Сбор данных об улицах из Wikidata

Мы используем SPARQL-эндпоинт Wikidata для поиска улиц, названных в честь исторических личностей, и сбора метаданных об этих личностях (пол, профессия, годы жизни).

## Исследовательские вопросы:
1. **Гендерный дисбаланс**: Какое соотношение мужских и женских имен в названиях улиц?
2. **Профессиональный портрет**: Улицы каких категорий деятелей (военные, писатели, политики, ученые) преобладают?
3. **Хронотоп памяти**: Представители каких исторических эпох наиболее ярко запечатлены в городской среде?

In [ ]:
import os
import re
import json
import requests
import pandas as pd

WIKIDATA_URL = "https://query.wikidata.org/sparql"

# Словарь для декодирования пола
GENDER_MAP = {
    "http://www.wikidata.org/entity/Q6581097": "Мужской",
    "http://www.wikidata.org/entity/Q6581072": "Женский"
}

### Вспомогательные функции для классификации профессий и эпох

In [ ]:
def categorize_occupation(label):
    label = str(label).lower()
    if any(w in label for w in ["писатель", "поэт", "литератор", "драматург", "баснописец", "прозаик"]):
        return "Литература"
    elif any(w in label for w in ["военный", "полководец", "маршал", "генерал", "адмирал", "офицер", "партизан", "герой советского союза"]):
        return "Военное дело"
    elif any(w in label for w in ["политик", "революционер", "государственный деятель", "император", "царь", "князь", "большевик", "декабрист"]):
        return "Политика и государство"
    elif any(w in label for w in ["учёный", "физик", "химик", "математик", "астроном", "географ", "историк", "академик", "биолог", "философ"]):
        return "Наука"
    elif any(w in label for w in ["художник", "композитор", "актер", "актриса", "режиссер", "скульптор", "архитектор", "певец", "музыкант"]):
        return "Искусство"
    elif any(w in label for w in ["космонавт", "астронавт"]):
        return "Космонавтика"
    elif any(w in label for w in ["путешественник", "мореплаватель", "исследователь"]):
        return "Путешествия"
    return "Другое"

def get_epoch(birth_year):
    if not birth_year or pd.isna(birth_year):
        return "Неизвестно"
    try:
        year = int(birth_year)
        if year < 1700:
            return "До XVIII века"
        elif 1700 <= year < 1800:
            return "XVIII век"
        elif 1800 <= year < 1917:
            return "XIX – начало XX века (Российская Империя)"
        elif 1917 <= year < 1991:
            return "Советский период"
        else:
            return "Постсоветский период"
    except ValueError:
        return "Неизвестно"

### Функция сбора данных из Wikidata через SPARQL

In [ ]:
def get_wikidata_streets(city_qid, city_name):
    print(f"Запрашиваем данные для {city_name}...")
    
    # SPARQL-запрос
    query = f"""
    SELECT DISTINCT ?street ?streetLabel ?coord ?namedAfter ?namedAfterLabel ?gender ?birth ?death ?occupationLabel WHERE {{
      ?street wdt:P131* wd:{city_qid};
              wdt:P625 ?coord;
              wdt:P138 ?namedAfter.
      ?namedAfter wdt:P21 ?gender.
      
      OPTIONAL {{ ?namedAfter wdt:P569 ?birth. }}
      OPTIONAL {{ ?namedAfter wdt:P570 ?death. }}
      OPTIONAL {{ ?namedAfter wdt:P106 ?occupation. }}
      
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "ru,en". }}
    }}
    LIMIT 2000
    """
    
    headers = {
        "User-Agent": "CityMemoryDHProject/1.0 (ostvald.anastasia@example.com)",
        "Accept": "application/sparql-results+json"
    }
    
    response = requests.get(WIKIDATA_URL, params={"query": query}, headers=headers, timeout=60)
    if response.status_code != 200:
        raise Exception(f"Ошибка API: {response.status_code}")
        
    data = response.json()
    results = data["results"]["bindings"]
    print(f"Получено {len(results)} записей.")

    rows = []
    for r in results:
        street_name = r.get("streetLabel", {}).get("value")
        coord_str = r.get("coord", {}).get("value")
        person_name = r.get("namedAfterLabel", {}).get("value")
        gender_url = r.get("gender", {}).get("value")
        birth = r.get("birth", {}).get("value")
        death = r.get("death", {}).get("value")
        occ = r.get("occupationLabel", {}).get("value", "")
        wikidata_url = r.get("namedAfter", {}).get("value")
        
        lat, lon = None, None
        if coord_str and "Point" in coord_str:
            match = re.search(r"Point\(([-\d\.]+) ([-\d\.]+)\)", coord_str)
            if match:
                lon = float(match.group(1))
                lat = float(match.group(2))

        birth_year = int(re.match(r"[+-]?(\d{{4}})", birth).group(1)) if birth and re.match(r"[+-]?(\d{{4}})", birth) else None
        death_year = int(re.match(r"[+-]?(\d{{4}})", death).group(1)) if death and re.match(r"[+-]?(\d{{4}})", death) else None
        
        rows.append({
            "street": street_name,
            "lat": lat,
            "lon": lon,
            "person": person_name,
            "gender": GENDER_MAP.get(gender_url, "Другое"),
            "birth_year": birth_year,
            "death_year": death_year,
            "occupation": occ,
            "wikidata_url": wikidata_url
        })
        
    df = pd.DataFrame(rows)
    
    grouped_rows = []
    for (street, person), group in df.groupby(["street", "person"]):
        lat = group["lat"].iloc[0]
        lon = group["lon"].iloc[0]
        gender = group["gender"].iloc[0]
        birth_year = group["birth_year"].dropna().iloc[0] if not group["birth_year"].dropna().empty else None
        death_year = group["death_year"].dropna().iloc[0] if not group["death_year"].dropna().empty else None
        wikidata_url = group["wikidata_url"].iloc[0]
        
        all_occupations = group["occupation"].dropna().unique()
        primary_occupation = "Другое"
        if len(all_occupations) > 0:
            categories = [categorize_occupation(o) for o in all_occupations]
            valid_cats = [c for c in categories if c != "Другое"]
            primary_occupation = valid_cats[0] if valid_cats else "Другое"
            
        epoch = get_epoch(birth_year)
        
        grouped_rows.append({
            "street": street,
            "lat": lat,
            "lon": lon,
            "person": person,
            "gender": gender,
            "birth_year": int(birth_year) if birth_year else None,
            "death_year": int(death_year) if death_year else None,
            "occupation": primary_occupation,
            "original_occupations": ", ".join(all_occupations),
            "epoch": epoch,
            "wikidata_url": wikidata_url
        })
    return pd.DataFrame(grouped_rows)

### Сбор данных и сохранение

In [ ]:
try:
    moscow_df = get_wikidata_streets("Q649", "Москва")
    spb_df = get_wikidata_streets("Q656", "Санкт-Петербург")
    
    # Выведем первые строки
    print("Москва:")
    print(moscow_df.head())
    
    # Сохраняем в JSON
    os.makedirs("data", exist_ok=True)
    moscow_df.to_json("data/moscow_streets.json", orient="records", force_ascii=False, indent=2)
    spb_df.to_json("data/spb_streets.json", orient="records", force_ascii=False, indent=2)
    print("Данные успешно сохранены в папку data!")
except Exception as e:
    print(f"Не удалось собрать данные из-за лимитов сети/API: {e}")
    print("В проекте используются предзагруженные качественные резервные наборы данных.")